# USGS Global Earthquake Forecasting Project
## Research Log and Reproducible Pipeline
Best current model: GCN (ROC-AUC ≈ 0.83)


## Dataset
USGS global earthquake catalog.
Experiments used M>=2.5 events (2000-2008) and later larger datasets.
Spatial grid: 2x2 degree cells.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


## Preprocessing
Convert time column, create spatial cells, aggregate by day.


In [ ]:
df['time']=pd.to_datetime(df['time'])
df['cell_lat']=(df['latitude']//2)*2
df['cell_lon']=(df['longitude']//2)*2
df['cell_id']=df['cell_lat'].astype(str)+'_'+df['cell_lon'].astype(str)


## Daily Aggregation Features
n_events, max_mag, mean_mag, mean_depth


In [ ]:
daily=df.groupby(['cell_id',pd.Grouper(key='time',freq='D')]).agg(n_events=('mag','count'),max_mag=('mag','max'),mean_mag=('mag','mean'),mean_depth=('depth','mean')).reset_index()


## Rolling Features
n_7d, n_30d, mag_max_7d, mag_max_30d, mag_mean_30d, activity_ratio


## Label
Predict whether a significant future event occurs in the prediction horizon.
Observed class ratio roughly 5% positives.


## Baseline Results
XGBoost ROC-AUC ≈ 0.78
Most important features: mag_max_7d, mag_max_30d, mag_mean_30d, n_7d, n_30d


## Graph Construction
Nodes = spatial cells
Edges = neighboring cells
Normalized adjacency A_norm used for message passing.


In [ ]:
class GCN(nn.Module):
    def __init__(self,in_feats,hidden=32):
        super().__init__()
        self.w1=nn.Linear(in_feats,hidden)
        self.w2=nn.Linear(hidden,1)
    def forward(self,x,A):
        x=torch.matmul(A,x)
        x=F.relu(self.w1(x))
        x=torch.matmul(A,x)
        x=torch.sigmoid(self.w2(x))
        return x.squeeze()


## Current Best Result
GCN ROC-AUC = 0.8317


## Next Research Steps
1. Leakage testing
2. Spatio-temporal GNN
3. Graph Attention Networks
4. Temporal Transformer on graph sequences
5. Paper preparation
